Vi skal i denne opgave finde den bedste løsning til at identificere og håndtere fake news. Jeg har i den forbindelse gået forskelligt materiale igennem og har på den måde undersøgt de forskellige muligheder. Jeg har læst artikler og prøvet at indsamle en så bred vifte af information som muligt. 

Men hvad er den bedste løsning til fake news detection? Når der bliver sagt “bedste løsning”, så er det lidt en fælde. For det kommer an på hvad vi måler på (accuracy, F1, generalization), og hvor realistisk det er at bygge/aflevere inden for tid og compute. Mange artikler viser nemlig, at man godt kan få mega høje tal på et datasæt, men så falder det når man tester på noget nyt (nye topics, ny skrivestil, nyt site osv.).

# Ide ud fra Algoritmer og datastruktur og Grundlæggende data science.

Ud fra vores AD materiale vurdere jeg at vi kan bruge nogle af de algoritmer vi har lært i undervisningen. Jeg har øje på blandt andet TF-IDF + Linear SVM.
Disse er skabt til at løse ligne problemer og kan som udgangspunkt blive brugt som baseline til tekstklassifikation. Flere af de artikler jeg har læst peger også på præcis dette. Mange systemer til fake news detection starter netop med TF-IDF features og en klassifikationsmodel som SVM eller lignende.
Derfor forstiller jeg mig at vi kan bygee pipelinen via disse algoritmer.


# Pipline og opsætning

Ud fra alt det materiale jeg har læst virker det til at den mest typiske løsning til fake news detection ser nogenlunde sådan ud:

## 1 Data collection

samle nyhedsartikler med labels fake / real

typiske datasæt kan f.eks. være:

* FakeNewsNet
* Kaggle fake news datasets
* scraped nyhedsartikler
* de angivet datasæt fra DIKU
* de artikler jeg snakker lidt om under kilder.

målet er at få et datasæt hvor hver artikel har en label. Sådan at vi ved hvad er sandt og falsk. Men vores model ved det ikke og skal selv vurdere. Sådan at vi får nogle samlinige resultater. 


## 2 Data Preprocessing

teksten renses og standardiseres så modellen kan arbejde med den.

typiske trin er:

* tekst rensning
* lowercase
* fjerne punctuation
* tokenization
* stopwords removal
* stemming eller lemmatization

nogle pipelines erstatter, ting som vi har gjort i tidligere opgaver.

* URL → `<URL>`
* tal → `<NUM>`
* datoer → `<DATE>`

## 3 Data splitting

training bruges til at træne modellen
validation bruges til at vælge hyperparametre
test bruges kun til at måle den endelige performance

Dette er vigtigt for at sikre at modellen kan generalisere til nye data. 

Vi skal have fokus på at der er nogle klassiske problemer som nemt kan opstå. 

**Data leakage:** Problemet opstår når datapunkter ikke er uafhængige.
**Random split:** Den simpleste metode er: shuffle datasættet split i train / validation / test

før modellen trænes deles datasættet i tre dele:

* training data (60–80%)
* validation data (10–20%)
* test data (10–20%)

**Samme data:** Hvis man scraper nyheder kan datasættet indeholde mange artikler fra samme kilde. Ideen er

CNN article
CNN article
CNN article
BBC article
BBC article

Hvis CNN-artikler ligger i både train og test kan modellen lære en bestemt stil i stedet for sandhed.

**Tidsdata:** Nyheder er også tidsafhængige.

Hvis man blander artikler fra 2016–2024 tilfældigt kan modellen lære mønstre fra fremtiden. Men i virkeligheden vil modellen blive brugt på fremtidige artikler.

**Potientiel løsnig:**
Hvis datasættet er lille kan man bruge k-fold cross validation.
Ideen:

* split data i k dele
* træne på k-1 dele
* teste på den sidste
* gentage k gange

Men også her skal man være forsigtig med leakage. Folds skal stadig respektere struktur i data (tid, grupper osv.). 

Formålet er at sikre at modellen evalueres på data der virkelig er ukendt for modellen, så resultatet bedre afspejler hvordan modellen vil fungere i praksis.


## 4 Feature representation

Først konverterer vi tekst til tal. Dette forstiller jeg mig at vi gør med den mest klassiske metode er:

TF-IDF (Term Frequency (TF) - Inverse Document Frequency (IDF))

Ideen bag er formlen her:

$$
TFIDF(t,d)=TF(t,d)\cdot\log\frac{N}{df(t)}
$$

hvor

* $TF(t,d)$ = hvor ofte ordet forekommer i dokumentet
* $df(t)$ = hvor mange dokumenter ordet findes i
* $N$ = antal dokumenter

ideen er så at vi opdeler ord i forskellige værdier som i dette eksempel:

* ord som “the” → lav værdi
* ord som “conspiracy” → høj værdi

Dette gør at modellen fokuserer på de ord som faktisk giver information om teksten.

Mange artikler viser også at TF-IDF stadig er meget brugt i fake news detection, fordi det er en simpel måde at repræsentere tekst på, men stadig meget effektiv.

Andre typiske metoder er:

* Bag-of-Words
* Word embeddings (Word2Vec / GloVe)
* contextual embeddings (BERT)

klassiske ML-modeller bruger ofte TF-IDF, og er den jeg regner med at bruge. Men BERT virker også effektiv og god. Man kunne overveje mulighederne for at bruge begge hvis det ikke er for krævende.

## 5 Feature selection / dimensionality reduction

tekstdata giver meget høje dimensioner (tusindvis af ord).

derfor kan man reducere antallet af features ved:

* feature selection
* chi-square test
* mutual information
* regularization
* dimensionality reduction

Formålet er at beholde informative features og fjerne støj. 

## 6 Model

selve klassifikationsmodellen trænes på de features vi har konstrueret. For at kunne evaluere forskellige metoder kan vi træne flere modeller og sammenligne deres performance.

### Sanity check model

Først trænes en meget simpel model for at sikre at datasættet og feature-repræsentationen fungerer korrekt.
typiske modeller til dette er:

* Naive Bayes

Naive Bayes fungerer ofte overraskende godt til tekstklassifikation fordi modellen antager at ordene optræder uafhængigt af hinanden.
Denne model kan derfor bruges som et sanity check og baseline.

### Klassiske machine learning modeller

derefter kan mere kraftfulde modeller trænes på de samme features.

typiske modeller er:

* Logistic Regression
* Linear SVM

Disse modeller fungerer ofte meget godt sammen med TF-IDF features.

### Deep learning modeller

Noget andet som går igen i mange artikler er brugen af neurale netværk og transformer-modeller. Her lærer modellen selv sproglige repræsentationer direkte fra data. Et eksempel på modeller er:

* BERT
* RoBERTa
* XLNet

Disse modeller kan forstå semantiske og kontekstuelle relationer i tekst, hvilket gør dem meget stærke til opgaver som fake news detection.
I nogle løsninger bruger man også en hybrid model, hvor transformer-modellen først laver en avanceret repræsentation af teksten, og derefter bruges en klassifikationsmodel som f.eks.

* XGBoost
* SVM

til selve klassifikationen.
Dette viser at man kan lave en pipeline hvor:
tekst → embeddings/features → klassifikationsmodel

### Data imbalance

Et problem i fake news detection er ofte **class imbalance**.
Datasættet kan indeholde langt flere rigtige nyheder end falske.

Hvis modellen kun optimerer **accuracy**, kan den ende med at klassificere næsten alt som “real news” og stadig få en høj score.

Derfor bruges ofte **F1-score** til evaluering, fordi den tager højde for både precision og recall.

En anden løsning er at justere selve træningen af modellen ved at bruge **weighted loss**. Her gives større vægt til den klasse der forekommer sjældnere i datasættet. På den måde bliver fejl på fake news straffet mere under træningen.

Ideen kan skrives som

$$
L(y,\hat{y})=-w_k\log(\hat{y}_k)
$$

hvor vægten typisk vælges som

$$
w_k=\frac{1}{|y_k|}
$$

så klasser med færre eksempler får højere vægt.

### Baselines man næsten bør have med: Naive Bayes og “billige” checks
Jeg tænker at vi tester også bruger Naive Bayes som sanity check/baseline. Jeg ser i en del artikler at Naive Bayes, selvom antagelserne er forsimplede, ofte virker godt i praksis til dokumentklassifikation og spamfiltrering, og det er hurtigt og kræver ikke sindssyg meget data. Derfor kan dette bruges som hurtigt check. 

Så i jeg vil have i vores bedste løsning noget i retning af:

Naive Bayes      → sanity check
Logistic / SVM   → stærk baseline
BERT / DL model  → avanceret model

Det gør vores opgave mere robust, fordi vi ikke bare vælger en model uden at benchmarke.

## 7 Hyperparameter tuning

modellens parametre optimeres via:

* grid search
* random search
* cross-validation

det sker typisk på validation-datasættet.

## 8 Evaluation

Til sidst evalueres modellen på test-data. Her måles hvor godt modellen kan klassificere nye artikler som fake eller real. Da dette er en binær klassifikation, kan hvert resultat placeres i en såkaldt confusion matrix.

Der findes fire mulige udfald:

* **True Positive (TP)** – en fake artikel bliver korrekt klassificeret som fake
* **True Negative (TN)** – en rigtig artikel bliver korrekt klassificeret som real
* **False Positive (FP)** – en rigtig artikel bliver forkert klassificeret som fake
* **False Negative (FN)** – en fake artikel bliver forkert klassificeret som real

Ud fra disse værdier kan man beregne forskellige metrics.

### Accuracy

Accuracy måler hvor stor en andel af alle forudsigelser der er korrekte:

$$
accuracy=\frac{TP+TN}{TP+TN+FP+FN}
$$

Problemet er at accuracy kan være misvisende hvis datasættet er ubalanceret Hvis langt de fleste artikler er “real news”, kan en model få høj accuracy ved næsten altid at forudsige real.

vores forlæser siger at han mener at det ikke er godt at bruge til ubalancerede datasæt. Derfor som udgangspunkt mener han at vi ikke skal bruge det. 

### Precision

Precision måler hvor ofte modellen har ret når den siger at en artikel er fake:

$$
precision=\frac{TP}{TP+FP}
$$

Høj precision betyder få false positiver, altså at modellen sjældent kalder rigtige nyheder for fake.

### Recall

Recall måler hvor stor en andel af alle fake artikler modellen faktisk finder:

$$
recall=\frac{TP}{TP+FN}
$$

Høj recall betyder få falske negativer, altså at modellen sjældent overser fake news.

### Precision–Recall tradeoff

Der findes typisk et tradeoff mellem precision og recall. En model kan få høj recall ved at klassificere mange artikler som fake, men så falder precision fordi flere rigtige artikler bliver markeret forkert.

### F1-score

F1-score kombinerer precision og recall til et tal ved at tage det harmoniske gennemsnit:

$$
F1=2\frac{precision\cdot recall}{precision+recall}
$$

F1-score bruges ofte i fake news detection fordi datasættet kan være ubalanceret, og fordi målet kræver at både precision og recall er høje.

## 9 Deployment / usage 

vi kan teste vores model i praksis eventuelt ved at bruge vores tidligere opgave om at hente artikler fra BBC også teste på dem. Der var over 800 styks. 

* analyserer nye artikler
* klassificerer fake / real
* giver en sandsynlighed for fake news.











**https://arxiv.org/pdf/1905.04749**

Artiklen beskriver, at tidligere forskning i fake news-detektion ofte repræsenterer tekster ved hjælp af TF-IDF features og derefter klassificerer dem med machine learning-modeller som SVM eller neurale netværk. Disse metoder udnytter sproglige mønstre i teksten og har vist gode resultater til at identificere misinformation. Modellerne evalueres typisk med klassiske klassifikationsmål som accuracy og F1-score. Dette passer godt med min egen første ide om at bruge TF-IDF sammen med en SVM-model som en mulig løsning på opgaven.

**https://www.nature.com/articles/s41598-025-29942-y**

Artiklen præsenterer en metode til fake news detection hvor man kombinerer dybe neurale sprogmodeller med klassiske machine learning-metoder. Her bruges en transformer-model (RoBERTa) til at udtrække avancerede tekstrepræsentationer fra nyheder, hvorefter en klassifikationsmodel (XGBoost) afgør om artiklen er falsk eller ægte. Derudover anvendes også TF-IDF og metadata, så modellen både udnytter sproglige mønstre og kontekst i teksten.

Resultaterne viser, at denne kombination giver høj præcision på datasæt som PolitiFact og GossipCop, hvilket tyder på at hybridmodeller kan være meget effektive til fake news detection.

Dette passer godt med vores egen ide om først at repræsentere teksten som features og derefter bruge en stærk klassifikationsmodel. Artiklen understøtter dermed tanken om en pipeline hvor tekst først konverteres til numeriske features, hvorefter en model som SVM, boosting eller neurale netværk kan bruges til selve klassifikationen.

**https://arxiv.org/html/2412.14276v1**

Artiklen undersøger brugen af transformations baserede sprogmodeller til fake news detection. Her sammenlignes BERT-baserede encoder-modeller med store autoregressive sprogmodeller (LLMs) for at se, hvor godt de kan klassificere nyhedsartikler som falske eller ægte. Til dette introducerer forfatterne et datasæt, hvor labels først er genereret med GPT-4 og derefter verificeret af mennesker for at sikre høj kvalitet.

Resultaterne viser, at transformer-modeller kan udnytte semantiske mønstre i teksten til effektivt at identificere misinformation. Modellerne evalueres ved hjælp af klassiske klassifikationsmål, så deres performance kan sammenlignes systematisk.

Dette passer godt med ideen om at bruge mere avancerede NLP-modeller til fake news detection. Artiklen viser især, at transformer-baserede repræsentationer kan være stærke features til klassifikation, hvilket kan indgå i en pipeline hvor tekst først repræsenteres numerisk og derefter klassificeres med en machine learning-model.

**https://arxiv.org/html/2512.18533**

Artiklen undersøger udfordringer ved automatiseret detektion af politiske falske nyheder, med særligt fokus på hvor godt modeller kan generalisere til nye data. Studiet analyserer flere machine learning-modeller på LIAR-datasættet, som indeholder politiske udsagn klassificeret efter deres sandhedsværdi.

Resultaterne viser, at mange modeller opnår høj performance på træningsdata, men mister en del præcision når de anvendes på nye eller ændrede datasæt. Dette kaldes et generalization gap, hvor modellen overtilpasser til træningsdata og derfor har sværere ved at identificere misinformation i nye situationer.

Dette er relevant for opgaven, fordi det viser hvor vigtigt det er at evaluere modeller korrekt og sikre at de kan generalisere til nye nyheder. Artiklen understøtter derfor behovet for en robust model og gode evalueringsmetoder, når man udvikler en løsning til fake news detection.

**https://etasr.com/index.php/ETASR/article/view/10678**

Artiklen undersøger, hvordan transformer-baserede sprogmodeller kan bruges til fake news detection. Studiet sammenligner modeller som BERT, RoBERTa og XLNet, der trænes til at klassificere nyhedsartikler som sande eller falske ud fra sproglige mønstre i teksten.

For at reducere beregningsomkostninger anvender forfatterne tekst-opsummering, så modellerne arbejder med kortere versioner af artiklerne. Resultaterne viser, at RoBERTa, når den finjusteres på de opsummerede tekster, opnår den bedste præcision i klassifikationen. Derudover bruges GPT-2 til at generere syntetiske falske nyheder, som anvendes til at teste modellernes evne til at skelne mellem ægte og kunstigt indhold.

Dette viser, at transformer-modeller kan være meget effektive til fake news detection, fordi de kan fange semantiske og kontekstuelle relationer i teksten. Artiklen understøtter derfor ideen om at bruge avancerede NLP-modeller til feature-repræsentation og klassifikation i en løsning til opgaven.

**https://scikit-learn.org/stable/modules/naive_bayes.html**

Naive Bayes er en probabilistisk klassifikationsmetode baseret på Bayes’ teorem, hvor man antager at features er betinget uafhængige givet klassen. Det betyder, at hver feature bidrager uafhængigt til sandsynligheden for en bestemt klasse. Selvom denne antagelse er simpel, fungerer metoden ofte godt i praksis, især i tekstklassifikation som spamfiltrering og dokumentanalyse.

I tekstklassifikation repræsenteres dokumenter typisk som feature-vektorer, for eksempel med bag-of-words eller TF-IDF. Naive Bayes beregner derefter sandsynligheden for hver klasse givet dokumentets features og vælger den klasse med højest sandsynlighed.

Dette er relevant for opgaven, fordi Naive Bayes ofte bruges som baseline i tekstklassifikation. Det kan derfor være en simpel model til at sammenligne mere avancerede metoder som SVM eller neurale netværk med i fake news detection.

**https://www.kaggle.com/code/therealsampat/fake-news-detection**

Denne Kaggle-analyse viser et eksempel på, hvordan maskinlæring kan bruges til fake news detection. Her anvendes et datasæt af nyhedsartikler, hvor hver tekst er mærket som enten fake eller real. Før modellen trænes, gennemgår teksten en række forbehandlingstrin, hvor stopord fjernes og teksten omdannes til numeriske features ved hjælp af TF-IDF.

Derefter trænes en klassifikationsmodel, blandt andet Naive Bayes og logistisk regression, til at afgøre om en artikel er falsk eller ægte. Modellerne lærer mønstre i ordenes forekomst i teksten og kan dermed identificere sproglige karakteristika, der ofte forekommer i misinformation.

Dette viser en klassisk pipeline for tekstklassifikation, hvor tekst først konverteres til features og derefter klassificeres med en machine learning-model. Eksemplet understøtter derfor ideen om at bruge TF-IDF sammen med en klassifikationsmodel i en løsning til fake news detection.

**https://keras.io/examples/nlp/text_classification_from_scratch/**

Denne tutorial viser, hvordan man kan bygge en tekstklassifikationsmodel fra bunden ved hjælp af Keras og TensorFlow. Her gennemgår man en komplet pipeline, hvor rå tekst først forbehandles og konverteres til numeriske repræsentationer, så den kan bruges som input til en maskinlæringsmodel.

Teksten behandles med tokenization og vectorization, hvor hvert dokument omdannes til en sekvens af tal. Derefter trænes et neuralt netværk, som typisk består af en embedding-layer, der lærer en numerisk repræsentation af ordene, efterfulgt af neurale lag der finder mønstre i teksten.

Dette viser, hvordan deep learning kan bruges til tekstklassifikation, hvor modellen selv lærer sproglige mønstre direkte fra data. Eksemplet understøtter derfor ideen om, at neurale netværk kan være et alternativ til mere klassiske metoder som SVM eller Naive Bayes i fake news detection.

**https://www.itm-conferences.org/articles/itmconf/pdf/2023/06/itmconf_icdsac2023_03005.pdf**

Artiklen undersøger brugen af maskinlæring til fake news detection, hvor forskellige klassifikationsmodeller anvendes til at identificere misinformation i nyhedsartikler. Før modellerne trænes, gennemgår teksten en række forbehandlingstrin, hvor støj fjernes, og teksten omdannes til numeriske features, som kan bruges af maskinlæringsalgoritmer.

Derefter trænes modellerne til at skelne mellem ægte og falske nyheder ved at lære mønstre i tekstens struktur og ordvalg. Resultaterne viser, at maskinlæringsbaserede metoder kan opnå høj præcision i klassifikationsopgaven.

Dette understøtter ideen om en pipeline hvor tekst først forbehandles og konverteres til features, hvorefter en klassifikationsmodel bruges til at identificere fake news. Artiklen viser dermed, at kombinationen af tekstforbehandling, feature-ekstraktion og klassifikation er en effektiv tilgang til opgaven.

**https://www.mdpi.com/2073-431X/14/6/237**

Artiklen undersøger brugen af maskinlæring til fake news detection i online medier. Studiet analyserer, hvordan forskellige modeller kan identificere misinformation ved at udnytte sproglige mønstre i tekstdata. Før modellerne trænes, gennemgår teksten en række forbehandlingstrin, herunder rensning af tekst, tokenisering og omdannelse til numeriske features.

Derefter trænes flere klassifikationsmodeller til at afgøre, om en artikel er ægte eller falsk. Resultaterne viser, at især transformer-baserede modeller kan opnå høj præcision i klassifikationen.

Dette understøtter ideen om at bruge en pipeline med tekstforbehandling, feature-repræsentation og en klassifikationsmodel til fake news detection. Artiklen viser samtidig, at datakvalitet og gode features spiller en vigtig rolle for modellens performance.

**https://arxiv.org/pdf/2411.12703**

Artiklen undersøger brugen af store sprogmodeller (LLMs) til fake news detection. Formålet er at analysere, hvor godt transformer-baserede modeller kan identificere misinformation sammenlignet med mere klassiske maskinlæringsmetoder. Modellerne trænes på nyhedsartikler, hvor hver tekst er mærket som enten sand eller falsk.

Metoden udnytter de semantiske og kontekstuelle repræsentationer, som transformer-modeller lærer fra store tekstkorpora. På den måde kan modellerne identificere mere komplekse sproglige mønstre i teksten. Performance evalueres med klassiske klassifikationsmål som accuracy og F1-score.

Resultaterne viser, at store sprogmodeller kan opnå høj præcision i fake news detection, men at resultatet afhænger af datakvalitet og korrekt finjustering af modellen. Dette understøtter ideen om at bruge avancerede NLP-modeller til tekstklassifikation, hvor transformer-baserede repræsentationer kan være stærke features i en løsning til opgaven.







# sidder til at finde fake news der altid skal kategorisers som fake. 

Artikler fra kendte misinformation-sider kan bruges som udgangspunkt for den fake klasse, men labels bør verificeres, og datasættet skal konstrueres sådan, at modellen ikke blot lærer kildespecifikke mønstre. Men ideen er at blande artiker fra disse sider ind i data sættet som vi med 100% garenti ved er fake. 

**https://beforeitsnews.com/**

**https://www.naturalnews.com/**

**https://thepeoplesvoice.tv/**

**https://theonion.com/**

God ide:
at bruge artikler fra kendte misinformation-kilder som mulige kandidater til fake-klassen.

Dårlig ide:
at antage, at alle artikler fra de sider automatisk er fake, uden yderligere kontrol. Så bygger du et skævt datasæt, og modellen kan ende med at lave “website detection” i stedet for “fake news detection”. Det problem er blevet fremhævet i nyere arbejde om source-based labels og generaliserbarhed.

# sidder med Faktatjek. Sidder som burde kun have korrekte artikler da de er blevet tjekket er eksperter. 

**https://www.politifact.com/**

**https://www.snopes.com/**

**https://leadstories.com/**

Dette skal dog tjekkes mere i dybden, om det er faktuelt korrekt at deres artikler er fakta tjekket og korrekte. 

Ideen:

Artikler og påstande kan labels ved hjælp af faktatjek-organisationer som PolitiFact eller Snopes. Disse organisationer undersøger systematisk nyheder og vurderer deres sandhedsværdi, hvilket gør deres vurderinger velegnede som ground-truth labels i maskinlæringsdatasæt.

# Sidder med almindelige artikler

# Engelske sidder som samler artikler fra andre sidder

Google News
NewsNow
Allsides

Ideen er først at bruge disse sidder som så vil hente en masse artikler. Tænker at vi godt kan hente en hel masse fra disse sidder. Grunden er at de kommer til at gå igen på de andre sider under. Derigennem vil vi ikke have dem 2 gange og vi kan springe dem over. Så ligesom at vi har et tjek som gør at vi ikke får en artikel 2 gange. skal vi gøre det igen her. Men vi vil bruge disse aggregatorer som udgangspunkt hente måske 500 sidder per side. Også kun 100 per sidde efterfølgende. Men vi tænker at mange af artiklerne ville gå igen og derfor ikke blive tilføjet. 

Jeg tænker at vi hurtigt kan få fejl i vores træningsdata hvis vi har mange artikler om de samme begivenheder. Vi vil gerne undgå at den tænker at et emne om krig i mellemøsten altid er korrekt. Eller at alt der kommer fra BBC altid er korrekt. Så vi skal som udgangspunkt begrænse vores antal af artikler per side til noget omkring de 100 stykker. Derfor er mange domæner vigtige, at vi blander vores datasæt sådan at alle artikler fra en kilde ikke kommer efter hinanden. 

## Engelske sidder der er good to go. 
BBC News
Reuters
AP News
The Guardian
Agence France-Presse
Bloomberg News
The New York Times
The Washington Post
The Wall Street Journal
The Telegraph
Financial Times
PBS NewsHour
NPR
Deutsche Welle
CNN
Sky News
Al Jazeera English

## Danske sidder, hvis de skal bruges skal koden ændres og tilpasses. 

Der forkommer hurtigt et problem med sprog hvis vi bruger danske nyhedssider så derfor anbefaler jeg det ikke. 

DR Nyheder
TV 2 Nyheder
Politiken
Berlingske
Jyllands-Posten